In [1]:
import numpy as np
from pathlib import Path
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# TensorFlow for NN surrogate (F8)
import tensorflow as tf
from tensorflow import keras
tf.get_logger().setLevel('ERROR')  # Suppress TF warnings

DATA_DIR = Path("..") / "initial_data"

def suggest_ucb_point(
    X,
    y,
    beta=1.5,
    n_candidates=10_000,
    random_state=0,
    bias_point=None,
    bias_scale=0.1,
    bias_fraction=0.5,
    kernel_type="rbf",
    n_restarts=5
):
    """
    Given current data (X, y) for ONE function, suggest the next query point x* using
    a Gaussian Process + Upper Confidence Bound (UCB) heuristic.

    Parameters
    ----------
    X : np.ndarray, shape (N, d)
        The inputs already tried for this function.
        N = number of past points, d = input dimension (2, 3, 4, ..., 8 here).
    y : np.ndarray, shape (N,)
        The outputs observed for those inputs (same order as rows in X).
    beta : float
        Controls exploration vs exploitation in UCB = mean + beta * std.
        - Larger beta = more exploration (try uncertain points).
        - Smaller beta = more exploitation (stay near current best).
    n_candidates : int
        Base number of random candidate points to try in [0, 1]^d.
        We'll possibly increase this for higher dimensions.
    random_state : int
        Seed for the random number generator so results are reproducible.
    bias_point : np.ndarray or None
        If provided, sample some candidates near this point (for exploitation).
    bias_scale : float
        Standard deviation for biased sampling around bias_point.
    bias_fraction : float
        Fraction of candidates to sample near bias_point (rest are uniform).
    kernel_type : str
        WEEK 7: Kernel selection as hyperparameter tuning.
        - "rbf" for smooth functions (F1, F3, F5, F8)
        - "matern" for rougher functions (F2, F4, F6, F7)
        WEEK 8: Reverted F2, F4, F6 to RBF after W7 regressions. Only F7 keeps Matern.
    n_restarts : int
        WEEK 7: Number of restarts for GP hyperparameter optimization.
        More restarts = better kernel hyperparameters, but slower.

    Returns
    -------
    best_x : np.ndarray, shape (d,)
        The chosen next input point (in [0, 1]^d) to submit as query.
    gp : GaussianProcessRegressor
        The fitted GP model, in case you want to inspect predictions later.
    """

    # --------------------------
    # 1. Basic info about X, y
    # --------------------------
    # X has shape (N, d): N = number of rows (past points), d = dimensions
    N, d = X.shape

    # --------------------------
    # 2. Build the GP kernel
    # --------------------------
    # WEEK 7 UPDATE: Kernel selection as hyperparameter tuning
    # - RBF: assumes infinitely differentiable (very smooth) functions
    # - Matern(nu=2.5): allows more roughness, better for jagged landscapes
    #   Think of it as adding regularisation that better fits noisy/rough functions.
    # WEEK 8 UPDATE: Matern caused regressions on F2, F4, F6. Reverted to RBF for those.
    # Only F7 keeps Matern (it worked there: 0.696 -> 1.510).
    if kernel_type == "matern":
        kernel = 1.0 * Matern(length_scale=np.ones(d), nu=2.5) + WhiteKernel(noise_level=1e-3)
    else:
        kernel = 1.0 * RBF(length_scale=np.ones(d)) + WhiteKernel(noise_level=1e-3)

    # --------------------------
    # 3. Create and fit the GP
    # --------------------------
    # WEEK 7 UPDATE: n_restarts_optimizer=5 for better kernel hyperparameter optimization
    # This helps the GP find better length-scales and noise levels.
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,   # Centers/scales y internally, helpful when magnitudes differ
        random_state=random_state,
        n_restarts_optimizer=n_restarts,
    )

    # Fit the GP on the current data (this is where it "learns" the function shape)
    gp.fit(X, y)

    # --------------------------
    # 4. Generate candidate points
    # --------------------------
    # We'll sample random points uniformly within [0, 1]^d.
    # For higher dimensions we increase the number of candidates because
    # the space is bigger and more "empty".
    if d <= 4:
        n = n_candidates
    else:
        # Double for d >= 5 to explore a bit more widely
        n = n_candidates * 2

    # modern NumPy RNG – local, reproducible
    rng = np.random.default_rng(random_state)

    if bias_point is not None:
        # Split candidates: some near bias_point, rest uniform
        n_biased = int(n * bias_fraction)
        n_uniform = n - n_biased
        
        # Biased candidates (normal distribution around bias_point)
        Xcand_biased = rng.normal(loc=bias_point, scale=bias_scale, size=(n_biased, d))
        Xcand_biased = np.clip(Xcand_biased, 0, 1)  # Keep in [0, 1]^d
        
        # Uniform candidates
        Xcand_uniform = rng.uniform(0.0, 1.0, size=(n_uniform, d))
        
        # Combine
        Xcand = np.vstack([Xcand_biased, Xcand_uniform])
    else:
        # Xcand has shape (n, d), with each coordinate between 0 and 1
        Xcand = rng.uniform(0.0, 1.0, size=(n, d))


    # --------------------------
    # 5. GP predictions at candidates
    # --------------------------
    # For each candidate point, we ask the GP for:
    #   - mu: predicted mean (what y value we expect)
    #   - std: predictive standard deviation (how uncertain we are)
    mu, std = gp.predict(Xcand, return_std=True)

    # --------------------------
    # 6. Compute UCB score
    # --------------------------
    # UCB(x) = mu(x) + beta * std(x)
    #   - mu(x): exploitation (high mean is good)
    #   - std(x): exploration (high uncertainty is good)
    # beta trades between them.
    ucb = mu + beta * std

    # --------------------------
    # 7. Mask already-seen points
    # --------------------------
    # WEEK 7 BUG FIX: Increased tolerance from 1e-8 to 1e-6
    # This prevents suggesting duplicate queries (like F7's W5/W6 duplicate).
    # The issue was that points effectively the same had tiny floating point differences.
    same_point = np.all(
        np.isclose(Xcand[:, None, :], X[None, :, :], atol=1e-6),
        axis=2,
    )  # shape: (n_candidates, N)
    mask_any_seen = np.any(same_point, axis=1)  # shape: (n_candidates,)
    ucb[mask_any_seen] = -np.inf

    # --------------------------
    # 8. Choose the best candidate
    # --------------------------
    # Take the candidate with the highest UCB value.
    best_idx = np.argmax(ucb)
    best_x = Xcand[best_idx]  # shape (d,)

    return best_x, gp


/opt/anaconda3/envs/tf_env/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
# ======================================
# Neural Network Surrogate (TensorFlow)
# ======================================
# For F8 (highest dimension), we use a small NN to propose
# candidates via gradient ascent, then let the GP decide via UCB.
# This combines NN expressiveness with GP uncertainty quantification.
#
# WEEK 7 UPDATE:
# - Increased n_random from 1000 to 2000 for wider random coverage
# - Added n_restarts for GP optimization
# - Fixed duplicate point masking (atol 1e-8 -> 1e-6)
#
# WEEK 8 UPDATE:
# - Reverted n_random from 2000 back to 1000 (W7's 2000 gave worst result: 9.660)
# - The extra random candidates may have diluted NN-guided candidates' influence
#
# WEEK 9 UPDATE:
# - W8's revert worked (9.660 -> 9.898). Keep n_random=1000.

def build_nn_surrogate(input_dim):
    """
    Build a small, regularized MLP surrogate model.
    Small network + dropout to avoid overfitting on tiny data.
    """
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dropout(0.1),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dropout(0.1),
        keras.layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
                  loss='mse')
    return model


def train_nn_surrogate(X, y, epochs=500, verbose=0):
    """
    Train the NN surrogate on observed data.
    Returns the trained model and normalization parameters.
    
    Uses Batch Gradient Descent (batch_size=len(X)) because:
    - We have tiny data (~40-50 points)
    - BGD provides stable, accurate gradient estimates
    - SGD would be too noisy with so few points
    """
    d = X.shape[1]
    
    # Normalize y for better training
    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_normalized = (y - y_mean) / y_std
    
    model = build_nn_surrogate(d)
    model.fit(X, y_normalized, epochs=epochs, batch_size=len(X), verbose=verbose)
    
    return model, y_mean, y_std


def gradient_ascent_on_nn(model, start_point, n_steps=50, lr=0.005, clip_min=0.05, clip_max=0.95):
    """
    Run gradient ascent on the NN surrogate to find a better point.
    Uses TensorFlow's GradientTape to compute gradients of output w.r.t. input.
    
    Conservative settings to avoid extreme boundary queries:
    - lr=0.005 (less aggressive than 0.01)
    - n_steps=50 (fewer steps toward boundaries)
    - clip to [0.05, 0.95] (avoid extreme boundaries)
    """
    # Start from the given point
    x = tf.Variable(start_point.reshape(1, -1), dtype=tf.float32)
    
    for _ in range(n_steps):
        with tf.GradientTape() as tape:
            y_pred = model(x, training=False)
        
        # Gradient of predicted output w.r.t. input
        grad = tape.gradient(y_pred, x)
        
        # Gradient ascent step (move in direction of increasing output)
        x.assign_add(lr * grad)
        
        # Clip to stay away from extreme boundaries [0.05, 0.95]
        x.assign(tf.clip_by_value(x, clip_min, clip_max))
    
    return x.numpy().flatten()


def suggest_nn_gp_hybrid(
    X,
    y,
    beta=1.5,
    n_random=1000,  # WEEK 8: Reverted from 2000 back to 1000
    n_gradient_steps=50,
    gradient_lr=0.005,
    clip_min=0.05,
    clip_max=0.95,
    top_k=3,
    random_state=0,
    n_restarts=5
):
    """
    NN proposes candidates via gradient ascent, GP selects via UCB.
    
    WEEK 7 UPDATE:
    - Increased n_random from 1000 to 2000 for wider coverage
    - Added n_restarts for better GP optimization
    - Fixed duplicate masking (atol 1e-8 -> 1e-6)
    
    WEEK 8 UPDATE:
    - Reverted n_random from 2000 back to 1000
    - W7's increase didn't help (9.660 was worst result)
    
    WEEK 9 UPDATE:
    - W8's revert worked (9.898). Keep settings unchanged.
    """
    N, d = X.shape
    rng = np.random.default_rng(random_state)
    
    # Step 1: Train NN surrogate
    tf.random.set_seed(random_state)
    model, y_mean, y_std = train_nn_surrogate(X, y, epochs=500, verbose=0)
    
    # Step 2: Gradient ascent from top-k points
    top_k_indices = np.argsort(y)[-top_k:]
    nn_candidates = []
    
    for idx in top_k_indices:
        start_point = X[idx].copy()
        optimized_point = gradient_ascent_on_nn(
            model, start_point, 
            n_steps=n_gradient_steps, 
            lr=gradient_lr,
            clip_min=clip_min,
            clip_max=clip_max
        )
        nn_candidates.append(optimized_point)
    
    nn_candidates = np.array(nn_candidates)
    
    # Step 3: Add random candidates (safety net)
    # WEEK 7: Increased from 1000 to 2000 for wider coverage
    # WEEK 8: Reverted back to 1000 (2000 gave worst result)
    # WEEK 9: Keep at 1000 (W8's revert worked: 9.898)
    random_candidates = rng.uniform(0.0, 1.0, size=(n_random, d))
    
    # Step 4: Combine all candidates
    all_candidates = np.vstack([nn_candidates, random_candidates])
    
    # Step 5: GP scores all candidates via UCB
    # Using RBF for F8 (high-dimensional but relatively smooth)
    kernel = 1.0 * RBF(length_scale=np.ones(d)) + WhiteKernel(noise_level=1e-3)
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=random_state,
        n_restarts_optimizer=n_restarts,  # WEEK 7: Better GP optimization
    )
    gp.fit(X, y)
    
    mu, std = gp.predict(all_candidates, return_std=True)
    ucb = mu + beta * std
    
    # WEEK 7 BUG FIX: Increased tolerance from 1e-8 to 1e-6
    same_point = np.all(
        np.isclose(all_candidates[:, None, :], X[None, :, :], atol=1e-6),
        axis=2,
    )
    mask_any_seen = np.any(same_point, axis=1)
    ucb[mask_any_seen] = -np.inf
    
    # Step 6: Return best according to GP
    best_idx = np.argmax(ucb)
    best_x = all_candidates[best_idx]
    
    # Track where the best candidate came from
    if best_idx < len(nn_candidates):
        source = f"NN gradient ascent (from top-{best_idx + 1} point)"
    else:
        source = "Random candidate"
    
    return best_x, gp, source


In [3]:
# ================================
# Load multi-round inputs/outputs
# ================================

def parse_multiline_lists(filepath):
    """
    Parse a file where each list may span multiple lines.
    Lists start with '[array(' or '[np.' and end with ')]'.
    """
    with open(filepath) as f:
        content = f.read()
    
    # Join all lines and split on list boundaries
    # Each complete entry starts with '[' and ends with ')]'
    rounds = []
    buffer = ""
    bracket_count = 0
    
    for char in content:
        buffer += char
        if char == '[':
            bracket_count += 1
        elif char == ']':
            bracket_count -= 1
            if bracket_count == 0 and buffer.strip():
                rounds.append(buffer.strip())
                buffer = ""
    
    return rounds

input_lines = parse_multiline_lists("inputs_m20.txt")
output_lines = parse_multiline_lists("outputs_m20.txt")

inputs_rounds = [eval(line, {"np": np, "array": np.array}) for line in input_lines]
outputs_rounds = [eval(line, {"np": np}) for line in output_lines]

n_rounds = len(inputs_rounds)
assert len(outputs_rounds) == n_rounds, "Mismatch between input/output rounds"

# Sanity: each round must have 8 entries (functions 1–8)
for r in range(n_rounds):
    assert len(inputs_rounds[r]) == 8, f"Round {r} inputs != 8"
    assert len(outputs_rounds[r]) == 8, f"Round {r} outputs != 8"

print(f"Loaded data from {n_rounds} previous week(s)")

# ============================
# Week 7 Configuration
# ============================
#
# HYPERPARAMETER TUNING FOCUS:
# This week we treat β, bias, and kernel choice as hyperparameters to tune.
#
# Global changes:
# - n_restarts_optimizer=5 for better GP kernel optimization
# - Matern(2.5) kernel for rough functions (F2, F4, F6, F7)
# - RBF kernel for smoother functions (F1, F3, F5, F8)
# - Fixed duplicate bug (atol 1e-8 -> 1e-6) that caused F7's W5/W6 repeat
#
# Key changes from Week 6:
# - F1: Try edge [0.05, 0.95] (corners and center all ≈0)
# - F2: Keep β=1.5, bias=0.4 + Matern (W6 recovered to 0.529)
# - F3: β=2.0, bias=0.4, RBF (W6's β=1.5/bias=0.5 regressed from -0.013 to -0.042)
# - F4: β=0.75, bias=0.6, Matern (slight regularisation against overfitting)
# - F5: Keep β=0.5, bias=0.7, RBF (working beautifully: 1999.95!)
# - F6: β=1.0, bias=0.3, Matern (compromise between 0.2 and 0.4)
# - F7: Keep β=2.0, bias=0.3, Matern + FIX DUPLICATE BUG
# - F8: Keep NN+GP, increase n_random to 2000
#
# ============================
# Week 8 Configuration
# ============================
#
# WEEK 7 RESULTS:
# - F5: 1999.95 → 4440.5 ✅ (boundary query [0,1,1,1] massive win!)
# - F7: 0.696 → 1.510 ✅ (finally beat initial 1.365! Bug fix + Matern worked)
# - F2, F3, F4, F6, F8: All regressed (Matern kernel may have overfit)
#
# WEEK 8 STRATEGY: Revert regressions, keep what worked
# - F1: Final corner [0.95, 0.95] (complete systematic coverage)
# - F2: β=2.0, bias=0.4, REVERT to RBF (Matern regressed 0.529→0.442; try more exploration)
# - F3: β=1.0, bias=0.5, RBF (exploit around W5's best -0.013)
# - F4: β=0.5, bias=0.6, REVERT to RBF (revert to W5 settings that gave 0.495)
# - F5: Keep β=0.5, bias=0.7, RBF (working! 4440.5)
# - F6: β=1.0, bias=0.4, REVERT to RBF (Matern regressed -0.313→-0.658)
# - F7: Keep β=2.0, bias=0.3, Matern (working! 1.510)
# - F8: Revert n_random to 1000 (2000 gave worst result 9.660)
#
# ============================
# Week 9 Configuration
# ============================
#
# WEEK 8 RESULTS:
# - F1: ≈0 (final corner confirmed flat — function is 0 everywhere)
# - F2: 0.442 → 0.587 ⬆️ (β=2.0 + RBF nearly matched initial 0.611!)
# - F3: -0.067 → -0.032 ⬆️ (exploitation helped, but W5's -0.013 still best)
# - F4: 0.269 → 0.444 ⬆️ (RBF revert worked, recovering toward 0.495)
# - F5: 4440.5 → 2235.7 ⬇️ (regressed — query had d4=0.715 instead of 1.0)
# - F6: -0.658 → -0.440 ⬆️ (RBF revert helped, but not back to -0.258)
# - F7: 1.510 → 1.508 ➖ (stable)
# - F8: 9.660 → 9.898 ⬆️ (n_random=1000 revert worked!)
#
# WEEK 9 STRATEGY: Hold steady, let GP refine. F1 changes (manual → GP-UCB), F5 explores.
# - F1: Switch to GP-UCB (all corners + center confirmed ≈0; let GP explore)
# - F2: Keep β=2.0, bias=0.4, RBF (0.587 nearly matched 0.611 — working!)
# - F3: Keep β=1.0, bias=0.5, RBF (continue exploiting toward -0.013)
# - F4: Keep β=0.5, bias=0.6, RBF (stay the course toward 0.495)
# - F5: SWITCH TO EXPLORATION! β=0.5→1.5, bias=0.7→0.4
#       REASON: Classmate achieved 5855 vs our 4440.5 (+32% ahead!)
#       This means [0,1,1,1] corner is NOT the global optimum.
#       We've been tunnel-visioning on this corner. Time to explore broadly.
# - F6: Keep β=1.0, bias=0.4, RBF (continue recovery toward -0.258)
# - F7: Keep β=2.0, bias=0.3, Matern (stable at 1.51 — don't touch)
# - F8: Keep n_random=1000 (9.898 recovery worked)

beta_map = {
    1: 1.5,    # F1 – finally use GP-UCB (function confirmed flat across domain)
    2: 2.0,    # F2 – keep same (0.587 nearly matched 0.611)
    3: 1.0,    # F3 – keep exploiting (improving toward -0.013)
    4: 0.5,    # F4 – keep same (recovering toward 0.495)
    5: 1.5,    # F5 – INCREASE from 0.5 to explore (classmate has 5855 vs our 4440.5!)
    6: 1.0,    # F6 – keep same (recovering toward -0.258)
    7: 2.0,    # F7 – keep same (stable at 1.51)
    8: 1.5,    # F8 – NN+GP hybrid
}

# Bias fraction per function (for biased sampling around best point)
# - Higher = more candidates near best point (exploitation)
# - Lower  = more uniform sampling (exploration)
bias_fraction_map = {
    1: 0.3,    # F1 – light bias (exploring flat function with GP)
    2: 0.4,    # F2 – keep same
    3: 0.5,    # F3 – keep same
    4: 0.6,    # F4 – keep same
    5: 0.4,    # F5 – DECREASE from 0.7 to explore broadly (classmate found better region)
    6: 0.4,    # F6 – keep same
    7: 0.3,    # F7 – keep same (working!)
    8: None,   # F8 – NN+GP hybrid
}

# WEEK 7 NEW: Kernel type per function
# - "matern" for rough/noisy functions (Matern nu=2.5 is less smooth than RBF)
# - "rbf" for smoother functions
# WEEK 8 UPDATE: Reverted F2, F4, F6 to RBF after W7 regressions
# Only F7 keeps Matern (it worked: 0.696 → 1.510)
# WEEK 9 UPDATE: Keep all kernels same (W8 reverts worked)
kernel_map = {
    1: "rbf",      # F1 – standard GP-UCB
    2: "rbf",      # F2 – keep RBF (W8: 0.587)
    3: "rbf",      # F3 – narrow optimum, relatively smooth
    4: "rbf",      # F4 – keep RBF (W8: 0.444)
    5: "rbf",      # F5 – unimodal, smooth
    6: "rbf",      # F6 – keep RBF (W8: -0.440)
    7: "matern",   # F7 – KEEP Matern (stable at 1.51)
    8: "rbf",      # F8 – NN+GP handles complexity
}


print(f"\n{'='*60}")
print(f"WEEK {n_rounds + 1} QUERIES")
print(f"{'='*60}")

for i in range(1, 9):
    # 1. Load initial data
    X_init = np.load(DATA_DIR / f"function_{i}" / "initial_inputs.npy")
    y_init = np.load(DATA_DIR / f"function_{i}" / "initial_outputs.npy")

    X = X_init.copy()
    y = y_init.copy()

    # 2. Append every round's (x, y) for this function
    for r in range(n_rounds):
        x_prev = np.array(inputs_rounds[r][i - 1])
        y_prev = float(outputs_rounds[r][i - 1])

        print(f"Week {r + 1} {x_prev} -> {y_prev}")

        X = np.vstack([X, x_prev.reshape(1, -1)])
        y = np.append(y, y_prev)

    # 3. Find best point so far
    best_idx = np.argmax(y)
    best_point = X[best_idx]
    best_y_so_far = y[best_idx]

    # 4. Generate query based on function-specific strategy
    if i == 1:
        # --------------------------
        # F1: Finally use GP-UCB
        # --------------------------
        # W4: [0.05, 0.05] → ≈0 (corner 1)
        # W5: [0.95, 0.05] → ≈0 (corner 2)
        # W6: [0.50, 0.50] → ≈0 (center)
        # W7: [0.05, 0.95] → ≈0 (corner 3)
        # W8: [0.95, 0.95] → ≈0 (corner 4 - final)
        # All corners + center confirmed ≈0. Function is flat.
        # Now let GP-UCB run to confirm and explore any remaining areas.
        best_x, gp = suggest_ucb_point(
            X, y,
            beta=beta_map[i],
            n_candidates=10_000,
            random_state=40 + i,
            bias_point=best_point,
            bias_scale=0.1,
            bias_fraction=bias_fraction_map[i],
            kernel_type=kernel_map[i],
            n_restarts=5,
        )
        strategy = f"GP-UCB (β={beta_map[i]}, bias={bias_fraction_map[i]}, kernel={kernel_map[i]})"
    
    elif i == 5:
        # --------------------------
        # F5: Manual exploration probe
        # --------------------------
        # W7: [0, 1, 1, 1] → 4440.5 (our best)
        # W8: [0, 1, 1, 0.715] → 2235.7 (regressed)
        # PROBLEM: Classmate reached 5855 by W2! We're stuck at a local optimum.
        # Our GP is anchored to [0,1,1,1] corner and won't explore elsewhere.
        # STRATEGY: Manually probe opposite corner for d1.
        # We've always pushed d1→0. Try d1=1 to see if there's a better basin.
        best_x = np.array([1.0, 1.0, 1.0, 1.0])
        strategy = "Manual probe [1,1,1,1] (exploring d1=1, classmate has 5855)"
    
    elif i == 8:
        # --------------------------
        # F8: NN + GP hybrid
        # --------------------------
        # WEEK 7: Increased n_random from 1000 to 2000 for wider coverage
        # WEEK 8: Reverted n_random back to 1000 (W7's 2000 gave worst result: 9.660)
        # WEEK 9: Keep n_random=1000 (W8's revert worked: 9.898)
        best_x, gp, source = suggest_nn_gp_hybrid(
            X, y,
            beta=beta_map[i],
            n_random=1000,  # WEEK 8: Reverted from 2000, WEEK 9: Keep
            n_gradient_steps=50,
            gradient_lr=0.005,
            clip_min=0.05,
            clip_max=0.95,
            top_k=3,
            random_state=40 + i,
            n_restarts=5,
        )
        strategy = f"NN+GP hybrid (n_random=1000, {source})"
    
    else:
        # --------------------------
        # F2-F4, F6-F7: GP-UCB with biased sampling
        # --------------------------
        # WEEK 7: Added kernel_type and n_restarts parameters
        # WEEK 8: Reverted F2, F4, F6 to RBF after W7 regressions
        # WEEK 9: Keep settings (W8 reverts worked)
        best_x, gp = suggest_ucb_point(
            X, y,
            beta=beta_map[i],
            n_candidates=10_000,
            random_state=40 + i,
            bias_point=best_point,
            bias_scale=0.1,
            bias_fraction=bias_fraction_map[i],
            kernel_type=kernel_map[i],
            n_restarts=5,
        )
        strategy = f"GP-UCB (β={beta_map[i]}, bias={bias_fraction_map[i]}, kernel={kernel_map[i]})"
    # 5. Format new query for the portal
    query_str = "-".join(f"{v:.6f}" for v in best_x)

    # WEEK 7 BUG FIX: Check if new query duplicates any existing point
    is_duplicate = np.any(np.all(np.isclose(X, best_x, atol=1e-6), axis=1))
    if is_duplicate:
        print("  ⚠️ WARNING: Suggested query is a DUPLICATE! Check masking logic.")

    # 6. Report
    print(f"Function {i}:")
    print(f"  Total points: {len(y)}")
    print(f"  Strategy: {strategy}")
    print(f"  Best y so far: {best_y_so_far:.6f}")
    print(f"  New query to submit: {query_str}")
    print()

Loaded data from 8 previous week(s)

WEEK 9 QUERIES
Week 1 [0.954151 0.767932] -> 9.14184425020275e-88
Week 2 [0.125971 0.826988] -> -7.249963615484764e-176
Week 3 [0.849072 0.349783] -> -4.116065399436474e-89
Week 4 [0.05 0.05] -> 7.570914060942952e-193
Week 5 [0.95 0.05] -> 4.406064392151042e-291
Week 6 [0.5 0.5] -> 2.6752879910742468e-09
Week 7 [0.05 0.95] -> 4.406064392151042e-291
Week 8 [0.95 0.95] -> 4.488845644107218e-144
Function 1:
  Total points: 18
  Strategy: GP-UCB (β=1.5, bias=0.3, kernel=rbf)
  Best y so far: 0.000000
  New query to submit: 0.703670-0.169871

Week 1 [0.773956 0.438878] -> 0.33661711576593556
Week 2 [0.858598 0.697368] -> 0.5919902522467393
Week 3 [0.094177 0.975622] -> -0.05158916960088751
Week 4 [0.733108 0.822566] -> 0.5873058139167443
Week 5 [0.777682 1.      ] -> 0.18668094922203354
Week 6 [0.507533 0.796346] -> 0.528986817634964
Week 7 [0.715421 0.89494 ] -> 0.4421444149323386
Week 8 [0.703661 0.926591] -> 0.5872224434256454
Function 2:
  Total poin